## Improved Feature Extraction (Center-Focused Bar Detection)
All bar features use the central region only.

In [ ]:
import os
import numpy as np
import pandas as pd
from astropy.io import fits
from scipy import ndimage
from scipy.stats import skew, kurtosis
from skimage.measure import moments, moments_central
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ── Center & radius ──────────────────────────────────────────────────────────
def get_center_and_radius(data):
    cy, cx = data.shape[0]//2, data.shape[1]//2
    y, x   = np.ogrid[:data.shape[0], :data.shape[1]]
    r      = np.sqrt((y-cy)**2 + (x-cx)**2)
    total  = data.sum()
    r_eff  = min(data.shape)//4
    for rad in range(5, min(data.shape)//2):
        if data[r < rad].sum() >= 0.5*total:
            r_eff = rad; break
    return cy, cx, r_eff


In [ ]:
# ── Bar features (central region only) ──────────────────────────────────────
def bar_bulge_ratio(data, cy, cx, r_eff):
    try:
        bar_r   = max(15, int(0.4*r_eff))
        bulge_r = max(5,  int(0.12*r_eff))
        y, x    = np.ogrid[:data.shape[0], :data.shape[1]]
        r       = np.sqrt((y-cy)**2 + (x-cx)**2)
        bulge_flux = data[r < bulge_r].sum()
        if bulge_flux < 1e-8: return 1.0
        bar_data = np.zeros_like(data)
        bar_data[(r >= bulge_r) & (r < bar_r)] = data[(r >= bulge_r) & (r < bar_r)]
        M = moments(bar_data, order=2)
        if M[0,0] < 1e-8: return 1.0
        xb, yb = M[1,0]/M[0,0], M[0,1]/M[0,0]
        mu = moments_central(bar_data, yb, xb, order=2)
        if mu[0,0] < 1e-8: return 1.0
        n20 = mu[2,0]/mu[0,0]; n02 = mu[0,2]/mu[0,0]; n11 = mu[1,1]/mu[0,0]
        t1  = (n20+n02)/2
        t2  = np.sqrt(((n20-n02)/2)**2 + n11**2)
        lp  = t1+t2; lm = t1-t2
        elong = np.sqrt(lp/lm) if lm > 1e-8 else 3.0
        elong = np.clip(elong, 1.0, 5.0)
        return np.clip(bar_data.sum()*elong/bulge_flux/2, 0.5, 10.0)
    except: return 1.0

def bar_fourier(data, cy, cx, r_eff):
    try:
        bar_r  = max(15, int(0.4*r_eff))
        inner  = max(5,  int(0.1*r_eff))
        y, x   = np.ogrid[:data.shape[0], :data.shape[1]]
        r      = np.sqrt((y-cy)**2 + (x-cx)**2)
        theta  = np.arctan2(y-cy, x-cx)
        ann    = (r >= inner) & (r < bar_r)
        if not np.any(ann): return 0.0
        I, phi = data[ann], theta[ann]
        A0 = I.sum()
        if A0 < 1e-8: return 0.0
        A2 = np.sqrt(np.sum(I*np.cos(2*phi))**2 + np.sum(I*np.sin(2*phi))**2) / A0
        return min(A2, 1.0)
    except: return 0.0

def central_asym(data, cy, cx, r_eff, angle=180):
    try:
        cr  = max(15, int(0.35*r_eff))
        ym, yM = max(0,cy-cr), min(data.shape[0],cy+cr)
        xm, xM = max(0,cx-cr), min(data.shape[1],cx+cr)
        c   = data[ym:yM, xm:xM]
        if c.size == 0: return 0.0
        rot = ndimage.rotate(c, angle, reshape=False, order=1)
        return min(np.sum(np.abs(c-rot)) / (2*np.sum(np.abs(c))+1e-10), 1.0)
    except: return 0.0


In [ ]:
# ── Unbarred-specific features ───────────────────────────────────────────────
def nuclear_conc(data, cy, cx, r_eff):
    try:
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r    = np.sqrt((y-cy)**2 + (x-cx)**2)
        nr   = max(3, int(0.05*r_eff))
        ro   = max(10, int(0.15*r_eff))
        n    = data[r <= nr].sum()
        ring = data[(r > nr) & (r <= ro)].sum()
        return min(n / (ring+1e-6), 5.0)
    except: return 0.0

def spiral_symmetry(data, cy, cx, r_eff):
    try:
        ir  = max(15, int(0.3*r_eff))
        orr = max(30, int(0.8*r_eff))
        y, x= np.ogrid[:data.shape[0], :data.shape[1]]
        r   = np.sqrt((y-cy)**2 + (x-cx)**2)
        th  = np.arctan2(y-cy, x-cx)
        reg = (r >= ir) & (r < orr)
        if not np.any(reg): return 0.0
        I, phi = data[reg], th[reg]
        A0 = I.sum() + 1e-10
        A2 = np.abs(np.sum(I*np.exp(2j*phi)))/A0
        A3 = np.abs(np.sum(I*np.exp(3j*phi)))/A0
        A4 = np.abs(np.sum(I*np.exp(4j*phi)))/A0
        return min((A3+A4)/(A2+1e-6), 10.0)
    except: return 0.0

def bar_ansae_test(data, cy, cx, r_eff):
    try:
        tr = max(10, int(0.25*r_eff))
        ss = max(3, tr//3)
        dirs = []
        if cy-ss >= 0 and cy+ss < data.shape[0]:
            if cx+tr+ss < data.shape[1]:
                dirs.append(data[cy-ss:cy+ss, cx+tr-ss:cx+tr+ss].mean())
            if cx-tr-ss >= 0:
                dirs.append(data[cy-ss:cy+ss, cx-tr-ss:cx-tr+ss].mean())
        if cx-ss >= 0 and cx+ss < data.shape[1]:
            if cy+tr+ss < data.shape[0]:
                dirs.append(data[cy+tr-ss:cy+tr+ss, cx-ss:cx+ss].mean())
            if cy-tr-ss >= 0:
                dirs.append(data[cy-tr-ss:cy-tr+ss, cx-ss:cx+ss].mean())
        if len(dirs) < 2: return 0.5
        mx, mn = max(dirs), np.mean(dirs)
        if mn < 1e-8: return 0.5
        return max(0.0, min(1.0 - (mx/mn - 1.0), 1.0))
    except: return 0.5

def edge_on_indicator(data, cy, cx):
    try:
        M  = moments(data, order=2)
        if M[0,0] < 1e-8: return 0.0
        xc, yc = M[1,0]/M[0,0], M[0,1]/M[0,0]
        mu = moments_central(data, yc, xc, order=2)
        if mu[0,0] < 1e-8: return 0.0
        m20, m02 = mu[2,0]/mu[0,0], mu[0,2]/mu[0,0]
        if m02 < 1e-8 or m20 < 1e-8: return 0.0
        ar = max(m20/m02, m02/m20)
        return max(0.0, min((ar-1.0)/5.0, 1.0))
    except: return 0.0


In [ ]:
# ── Standard morphology ──────────────────────────────────────────────────────
def concentration_C(data, cy, cx):
    try:
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r    = np.sqrt((y-cy)**2 + (x-cx)**2)
        tot  = data.sum(); r20=r80=0
        for rad in range(1, min(data.shape)//2):
            frac = data[r < rad].sum()/tot
            if frac >= 0.2 and r20 == 0: r20 = rad
            if frac >= 0.8: r80 = rad; break
        if r20 == 0 or r80 == 0 or r20 >= r80: return 0.0
        return max(0.0, 5*np.log10(r80/r20))
    except: return 0.0

def gini(data):
    try:
        flat = data.flatten(); flat = flat[flat > 0]
        if len(flat) == 0: return 0.0
        s = np.sort(flat); n = len(s); idx = np.arange(1,n+1)
        return max(0.0, min((2*np.sum(idx*s))/(n*np.sum(s)) - (n+1)/n, 1.0))
    except: return 0.0

def ellipticity(data, cy, cx):
    try:
        M  = moments(data, order=2)
        if M[0,0] < 1e-8: return 0.0
        xc, yc = M[1,0]/M[0,0], M[0,1]/M[0,0]
        mu = moments_central(data, yc, xc, order=2)
        if mu[0,0] < 1e-8: return 0.0
        m20 = mu[2,0]/mu[0,0]; m02 = mu[0,2]/mu[0,0]; m11 = mu[1,1]/mu[0,0]
        num = np.sqrt((m20-m02)**2 + 4*m11**2)
        return min(num / (m20+m02+1e-10), 1.0)
    except: return 0.0

def R50_R90(data, cy, cx):
    try:
        y, x = np.ogrid[:data.shape[0], :data.shape[1]]
        r    = np.sqrt((y-cy)**2 + (x-cx)**2)
        tot  = data.sum(); R50=R90=0
        for rad in range(1, min(data.shape)//2):
            frac = data[r < rad].sum()/tot
            if frac >= 0.5 and R50 == 0: R50 = rad
            if frac >= 0.9: R90 = rad; break
        return R50 or min(data.shape)//4, R90 or min(data.shape)//2
    except: return 0, 0


In [ ]:
# ── Main extraction ──────────────────────────────────────────────────────────
def extract_features(path):
    try:
        with fits.open(path) as hdul:
            data = hdul[0].data
        if data is None: return None
        data = data.astype(float)
        data = (data - data.min()) / (data.max() - data.min() + 1e-10)
        cy, cx, r_eff = get_center_and_radius(data)
        R50, R90 = R50_R90(data, cy, cx)
        return {
            'filename':                  os.path.basename(path),
            'Bar_Bulge_Ratio_Focused':   bar_bulge_ratio(data, cy, cx, r_eff),
            'Bar_Fourier_Strength_Focused': bar_fourier(data, cy, cx, r_eff),
            'Central_Asymmetry_180':     central_asym(data, cy, cx, r_eff, 180),
            'Central_Asymmetry_90':      central_asym(data, cy, cx, r_eff, 90),
            'Nuclear_Concentration_Ratio': nuclear_conc(data, cy, cx, r_eff),
            'Spiral_Symmetry_Score':     spiral_symmetry(data, cy, cx, r_eff),
            'Bar_Ansae_Test':            bar_ansae_test(data, cy, cx, r_eff),
            'Edge_On_Indicator':         edge_on_indicator(data, cy, cx),
            'Concentration':             concentration_C(data, cy, cx),
            'R50':                       R50,
            'R90':                       R90,
            'Gini':                      gini(data),
            'Ellipticity':               ellipticity(data, cy, cx),
            'Mean_Intensity':            data.mean(),
            'Std_Intensity':             data.std(),
            'Skewness':                  skew(data.flatten()),
            'Kurtosis':                  kurtosis(data.flatten()),
            'Radial_Profile_Ratio':      R90/(R50+1e-6) if R50 > 0 else 0,
            'Effective_Radius':          r_eff,
        }
    except Exception as e:
        print(f"Error {path}: {e}"); return None

def process_folder(folder, out_csv):
    files = [f for f in os.listdir(folder) if f.endswith('.fits')]
    print(f"Found {len(files)} FITS files in {folder}")
    recs = []
    for i, f in enumerate(files):
        if (i+1) % 50 == 0: print(f"  {i+1}/{len(files)}")
        feat = extract_features(os.path.join(folder, f))
        if feat: recs.append(feat)
    df = pd.DataFrame(recs)
    cols = ['filename'] + [c for c in df.columns if c != 'filename']
    df[cols].to_csv(out_csv, index=False)
    print(f"Saved {out_csv}  shape={df.shape}")
    return df

if __name__ == "__main__":
    print("=" * 80)
    print("IMPROVED FEATURE EXTRACTION – CENTER-FOCUSED BAR DETECTION")
    print("=" * 80)
    for folder, csv in [("train","train_galaxy_features_improved.csv"),
                        ("test", "test_galaxy_features_improved.csv")]:
        if os.path.exists(folder):
            process_folder(folder, csv)
        else:
            print(f"Warning: {folder} not found")
    print("\nDone!")
